### Dataset
The dataset we use to run the model is the one generated in the NLP notebook up to step 3, which cleans the data from stopwords, legal words from the Canadian and American bills.


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split
from collections import Counter

ROOT  = Path("..").resolve()
NORM  = ROOT / "data" / "normalized"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

ModuleNotFoundError: No module named 'torch'

### Load Data

In [ ]:
df = pd.read_csv(NORM / "bills_clean.csv")

# Drop rows without text or label
df = df[df["full_text_clean"].notna() & df["passed"].notna()].copy()
df["full_text_clean"] = df["full_text_clean"].astype(str)
df["passed"] = df["passed"].astype(int)

print(f"Total bills: {len(df):,}")
print(f"Pass rate:   {df['passed'].mean():.3f}")
print(f"Sources:     {df['source'].value_counts().to_dict()}")

### Tokenization & Vocabulary

In [ ]:
MAX_VOCAB  = 30_000
MAX_SEQ_LEN = 512
PAD_IDX    = 0
UNK_IDX    = 1

def tokenize(text: str) -> list[str]:
    return text.lower().split()

# Build vocabulary from training split only (fitted after split below)
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["passed"])
train_df, val_df  = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df["passed"])

counter = Counter()
for text in train_df["full_text_clean"]:
    counter.update(tokenize(text))

vocab = ["<PAD>", "<UNK>"] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f"Vocab size: {len(vocab):,}")
print(f"Train / Val / Test: {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")

### Dataset & DataLoaders

In [ ]:
class BillDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, word2idx: dict, max_len: int):
        self.labels = torch.tensor(dataframe["passed"].values, dtype=torch.float32)
        self.sequences = [
            torch.tensor(
                [word2idx.get(w, UNK_IDX) for w in tokenize(text)][:max_len],
                dtype=torch.long,
            )
            for text in dataframe["full_text_clean"]
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_fn(batch):
    seqs, labels = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    return seqs_padded, torch.stack(labels)


BATCH_SIZE = 32

train_loader = DataLoader(BillDataset(train_df, word2idx, MAX_SEQ_LEN), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(BillDataset(val_df,   word2idx, MAX_SEQ_LEN), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(BillDataset(test_df,  word2idx, MAX_SEQ_LEN), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")